# Clustering Countries on Sustainable Energy - Comprehensive AnalysisClustering Countries on Sustainable Energy Analysis====================================================This script performs a comprehensive analysis of sustainable energy patterns across countries.Google Colab Usage:-------------------1. Upload this script to Google Colab or run in a notebook cell2. Install required packages (see below)3. Run the script - it will automatically load data from GitHub or prompt for uploadRequired packages (install in Colab):!pip install pandas numpy matplotlib seaborn plotly scikit-learn statsmodels openpyxl country_converter kaleidoThe analysis consists of 4 parts:1. Energy Archetypes: KMeans clustering (k=4) with PCA visualization2. Price vs Efficiency: Electricity price and energy intensity analysis3. Leapfrogging: Developing economies renewable adoption analysis4. Momentum Analysis: Temporal trends in renewable adoption (2010-2020)Outputs:- CSV files: countries_clusters.csv, momentum_analysis_2010_2020.csv- Visualizations: PCA plots, boxplots, choropleths, scatter plots, histograms

In [ ]:
# Install required packages (run this cell first in Google Colab)!pip install pandas numpy matplotlib seaborn plotly scikit-learn statsmodels openpyxl country_converter kaleido -q

## Imports and Configuration

In [ ]:
import osimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.cluster import KMeansfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.metrics import silhouette_scorefrom sklearn.linear_model import LinearRegressionimport plotly.express as pximport plotly.graph_objects as gofrom scipy.stats import pearsonrimport country_converter as cocowarnings.filterwarnings('ignore')# ConfigurationRANDOM_STATE = 42N_CLUSTERS = 4OUTPUT_DIR = 'outputs'# Cluster labeling heuristics (configurable)CLUSTER_LABELS = {    'high_renewable_high_access': 'Green Leaders',    'high_fossil_high_consumption': 'Fossil Giants',    'moderate_renewable_developing': 'Emerging Transitioners',    'low_access_low_consumption': 'Energy Deficient'}# Momentum analysis thresholds (configurable)MOMENTUM_THRESHOLDS = {    'renewable_slope_high': 0.5,  # %/year growth    'renewable_slope_low': 0.1,    'co2_slope_high': -0.02,  # negative = reduction}# Create output directory if it doesn't existos.makedirs(OUTPUT_DIR, exist_ok=True)

## Part 1: Energy Archetypes

In [ ]:
print("\n" + "=" * 80)print("PART 1: THE FOUNDATION - ENERGY ARCHETYPES")print("=" * 80)print()print("Building feature set for clustering...")# Start with OWID dataclustering_data = owid_latest[['country', 'iso_code']].copy()# Feature 1: Renewable share (from OWID)if 'renewables_share_energy' in owid_latest.columns:    clustering_data['renewable_share'] = owid_latest['renewables_share_energy']elif 'renewables_electricity' in owid_latest.columns and 'electricity_generation' in owid_latest.columns:    clustering_data['renewable_share'] = (        owid_latest['renewables_electricity'] / owid_latest['electricity_generation'] * 100    )else:    clustering_data['renewable_share'] = np.nan# Feature 2: Fossil electricity share (from OWID)if 'fossil_share_energy' in owid_latest.columns:    clustering_data['fossil_share'] = owid_latest['fossil_share_energy']elif 'fossil_electricity' in owid_latest.columns and 'electricity_generation' in owid_latest.columns:    clustering_data['fossil_share'] = (        owid_latest['fossil_electricity'] / owid_latest['electricity_generation'] * 100    )else:    clustering_data['fossil_share'] = np.nan# Feature 3: Energy consumption per capita (from OWID)if 'energy_per_capita' in owid_latest.columns:    clustering_data['consumption_per_capita'] = owid_latest['energy_per_capita']elif 'primary_energy_consumption' in owid_latest.columns and 'population' in owid_latest.columns:    clustering_data['consumption_per_capita'] = (        owid_latest['primary_energy_consumption'] / owid_latest['population']    )else:    clustering_data['consumption_per_capita'] = np.nan# Feature 4: Access to electricity (from global data)# Merge with global dataglobal_data_merge = global_data_latest[['Entity', 'Access to electricity (% of population)']].copy()global_data_merge.columns = ['country', 'access_to_electricity']clustering_data = clustering_data.merge(global_data_merge, on='country', how='left')# Try to load Getting Electricity data for additional access infotry:    electricity_access = load_data_from_github_or_local('Getting Electricity.xlsx')    print(f"  Loaded Getting Electricity data: {electricity_access.shape}")    # Merge if useful columns exist    if 'Economy' in electricity_access.columns:        access_cols = [c for c in electricity_access.columns if 'access' in c.lower() or 'reliability' in c.lower()]        if access_cols:            electricity_access_merge = electricity_access[['Economy'] + access_cols[:1]].copy()            electricity_access_merge.columns = ['country'] + ['electricity_reliability']            clustering_data = clustering_data.merge(electricity_access_merge, on='country', how='left')except Exception as e:    print(f"  Could not load Getting Electricity data: {e}")# Add GDP per capita for contextif 'gdp' in owid_latest.columns and 'population' in owid_latest.columns:    clustering_data['gdp_per_capita'] = owid_latest['gdp'] / owid_latest['population']else:    clustering_data['gdp_per_capita'] = np.nanprint(f"\nFeatures created: {list(clustering_data.columns)}")print(f"\nData shape: {clustering_data.shape}")print(f"\nMissing values:\n{clustering_data.isnull().sum()}")# Select core features for clusteringfeature_columns = ['renewable_share', 'fossil_share', 'consumption_per_capita', 'access_to_electricity']print(f"\nCore clustering features: {feature_columns}")# Remove rows with missing values in core featuresclustering_clean = clustering_data.dropna(subset=feature_columns).copy()print(f"Countries with complete data: {len(clustering_clean)}")# Prepare features for clusteringX = clustering_clean[feature_columns].valuesscaler = StandardScaler()X_scaled = scaler.fit_transform(X)print("\n--- Running KMeans Clustering (k=4) ---")kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)clustering_clean['cluster'] = kmeans.fit_predict(X_scaled)# Compute silhouette scoresilhouette = silhouette_score(X_scaled, clustering_clean['cluster'])print(f"\nSilhouette Score: {silhouette:.3f}")# Compute PCA for visualizationpca = PCA(n_components=2, random_state=RANDOM_STATE)X_pca = pca.fit_transform(X_scaled)clustering_clean['pca1'] = X_pca[:, 0]clustering_clean['pca2'] = X_pca[:, 1]print(f"PCA explained variance: {pca.explained_variance_ratio_}")print(f"  Component 1: {pca.explained_variance_ratio_[0]:.1%}")print(f"  Component 2: {pca.explained_variance_ratio_[1]:.1%}")# Heuristic cluster labeling based on feature meansprint("\n--- Cluster Characteristics ---")cluster_stats = clustering_clean.groupby('cluster')[feature_columns].mean()print(cluster_stats)# Assign labels based on characteristicsdef assign_cluster_label(row):    """Heuristic labeling based on cluster characteristics"""    renewable = row['renewable_share']    fossil = row['fossil_share']    consumption = row['consumption_per_capita']    access = row['access_to_electricity']        if renewable > 40 and access > 95:        return 'Green Leaders'    elif fossil > 60 and consumption > 50000:        return 'Fossil Giants'    elif access < 80 and consumption < 20000:        return 'Energy Deficient'    else:        return 'Emerging Transitioners'cluster_labels_map = {}for cluster_id in range(N_CLUSTERS):    cluster_mean = cluster_stats.loc[cluster_id]    label = assign_cluster_label(cluster_mean)    cluster_labels_map[cluster_id] = labelprint("\n--- Cluster Labels ---")for cluster_id, label in cluster_labels_map.items():    print(f"Cluster {cluster_id}: {label}")clustering_clean['cluster_label'] = clustering_clean['cluster'].map(cluster_labels_map)# Save cluster assignmentsoutput_path = os.path.join(OUTPUT_DIR, 'countries_clusters.csv')clustering_clean[['country', 'iso_code', 'cluster', 'cluster_label'] + feature_columns].to_csv(    output_path, index=False)print(f"\n✓ Saved cluster assignments to {output_path}")# Visualization 1: PCA Scatter Plotprint("\n--- Creating Visualizations ---")fig = px.scatter(    clustering_clean,    x='pca1',    y='pca2',    color='cluster_label',    hover_data=['country', 'renewable_share', 'fossil_share', 'consumption_per_capita', 'access_to_electricity'],    title=f'Energy Archetypes: PCA Visualization (k={N_CLUSTERS}, Silhouette={silhouette:.3f})',    labels={'pca1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',            'pca2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'},    width=1000,    height=600)fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color='DarkSlateGrey')))fig.write_html(os.path.join(OUTPUT_DIR, 'pca_clusters.html'))print("  ✓ PCA scatter plot saved")# Try to save as PNG (requires kaleido)try:    fig.write_image(os.path.join(OUTPUT_DIR, 'pca_clusters.png'), width=1000, height=600)    print("  ✓ PCA scatter plot PNG saved")except Exception as e:    print(f"  ✗ Could not save PNG: {e}")# Visualization 2: Boxplots for features by clusterfig, axes = plt.subplots(2, 2, figsize=(14, 10))fig.suptitle('Feature Distributions by Cluster', fontsize=16, y=1.00)for idx, feature in enumerate(feature_columns):    ax = axes[idx // 2, idx % 2]    data_to_plot = [clustering_clean[clustering_clean['cluster_label'] == label][feature].values                    for label in cluster_labels_map.values()]    bp = ax.boxplot(data_to_plot, labels=list(cluster_labels_map.values()), patch_artist=True)    ax.set_title(feature.replace('_', ' ').title())    ax.set_ylabel('Value')    ax.tick_params(axis='x', rotation=45)        # Color boxes    colors = px.colors.qualitative.Plotly[:N_CLUSTERS]    for patch, color in zip(bp['boxes'], colors):        patch.set_facecolor(color)plt.tight_layout()plt.savefig(os.path.join(OUTPUT_DIR, 'feature_boxplots.png'), dpi=150, bbox_inches='tight')print("  ✓ Feature boxplots saved")plt.close()# Visualization 3: World Choropleth Map# Convert country names to ISO3 codescc = coco.CountryConverter()clustering_clean['iso3'] = clustering_clean['country'].apply(lambda x: cc.convert(names=x, to='ISO3'))fig = px.choropleth(    clustering_clean,    locations='iso3',    color='cluster_label',    hover_name='country',    hover_data={'renewable_share': ':.1f', 'fossil_share': ':.1f', 'access_to_electricity': ':.1f'},    title='Global Energy Archetypes by Country',    color_discrete_sequence=px.colors.qualitative.Plotly,    width=1200,    height=700)fig.update_geos(showcountries=True, countrycolor="lightgray")fig.write_html(os.path.join(OUTPUT_DIR, 'world_choropleth_clusters.html'))print("  ✓ World choropleth saved")try:    fig.write_image(os.path.join(OUTPUT_DIR, 'world_choropleth_clusters.png'), width=1200, height=700)    print("  ✓ World choropleth PNG saved")except Exception as e:    print(f"  ✗ Could not save PNG: {e}")# Part 1 Conclusionprint("\n" + "=" * 80)print("PART 1 CONCLUSION: Energy Archetypes")print("=" * 80)print(f"""The clustering analysis identified {N_CLUSTERS} distinct energy archetypes with a silhouette score of {silhouette:.3f}:""")for cluster_id, label in cluster_labels_map.items():    count = (clustering_clean['cluster'] == cluster_id).sum()    stats = cluster_stats.loc[cluster_id]    print(f"{label} (n={count}):")    print(f"  - Renewable share: {stats['renewable_share']:.1f}%")    print(f"  - Fossil share: {stats['fossil_share']:.1f}%")    print(f"  - Consumption per capita: {stats['consumption_per_capita']:.0f} kWh")    print(f"  - Access to electricity: {stats['access_to_electricity']:.1f}%")    print()print("These archetypes reveal distinct pathways in energy development and provide a framework")print("for understanding global energy transitions. The PCA visualization shows clear separation")print("between clusters, suggesting meaningful differences in energy profiles.")print()

## Part 2: Price vs Efficiency

In [ ]:
print("\n" + "=" * 80)print("PART 2: THE ECONOMIC DRIVER - PRICE VS EFFICIENCY")print("=" * 80)print()try:    # Load electricity price data    price_data = load_data_from_github_or_local('P_Electric Prices by Country.xlsx')    print(f"Electricity price data shape: {price_data.shape}")    print(f"Columns: {list(price_data.columns)}")        # Find price column (USD/kWh or similar)    price_cols = [c for c in price_data.columns if 'price' in c.lower() or 'usd' in c.lower() or 'kwh' in c.lower()]    country_col = [c for c in price_data.columns if 'country' in c.lower() or 'nation' in c.lower() or 'economy' in c.lower()][0]        if price_cols:        price_col = price_cols[0]    else:        # Use second column as price if no obvious price column        price_col = price_data.columns[1]        print(f"Using country column: {country_col}")    print(f"Using price column: {price_col}")        # Prepare price data    price_merge = price_data[[country_col, price_col]].copy()    price_merge.columns = ['country', 'electricity_price_usd_kwh']    price_merge['electricity_price_usd_kwh'] = pd.to_numeric(price_merge['electricity_price_usd_kwh'], errors='coerce')        # Merge with clustering data    price_analysis = clustering_clean.merge(price_merge, on='country', how='inner')    print(f"\nCountries with price data: {len(price_analysis)}")        # Get energy intensity from global data (if available)    if 'Energy intensity level of primary energy (MJ/$2017 PPP GDP)' in global_data_latest.columns:        intensity_merge = global_data_latest[['Entity', 'Energy intensity level of primary energy (MJ/$2017 PPP GDP)']].copy()        intensity_merge.columns = ['country', 'energy_intensity']        price_analysis = price_analysis.merge(intensity_merge, on='country', how='left')    else:        # Use consumption per capita as proxy for intensity        price_analysis['energy_intensity'] = price_analysis['consumption_per_capita']        # Remove missing values    price_analysis_clean = price_analysis.dropna(subset=['electricity_price_usd_kwh', 'energy_intensity'])    print(f"Countries with complete price and intensity data: {len(price_analysis_clean)}")        if len(price_analysis_clean) > 10:        # Simple regression: Price vs Energy Intensity        X_price = price_analysis_clean['electricity_price_usd_kwh'].values.reshape(-1, 1)        y_intensity = price_analysis_clean['energy_intensity'].values                lr_simple = LinearRegression()        lr_simple.fit(X_price, y_intensity)        y_pred_simple = lr_simple.predict(X_price)                r2_simple = lr_simple.score(X_price, y_intensity)        corr, p_value = pearsonr(X_price.flatten(), y_intensity)                print(f"\n--- Simple Regression: Price vs Energy Intensity ---")        print(f"Correlation: {corr:.3f} (p={p_value:.4f})")        print(f"R²: {r2_simple:.3f}")        print(f"Coefficient: {lr_simple.coef_[0]:.2f}")        print(f"Intercept: {lr_simple.intercept_:.2f}")                # Visualization: Scatter with trendline        fig = px.scatter(            price_analysis_clean,            x='electricity_price_usd_kwh',            y='energy_intensity',            color='cluster_label',            hover_data=['country'],            title=f'Electricity Price vs Energy Intensity (R²={r2_simple:.3f}, p={p_value:.4f})',            labels={'electricity_price_usd_kwh': 'Electricity Price (USD/kWh)',                    'energy_intensity': 'Energy Intensity'},            trendline='ols',            width=900,            height=600        )        fig.write_html(os.path.join(OUTPUT_DIR, 'price_vs_intensity.html'))        print("  ✓ Price vs intensity plot saved")                try:            fig.write_image(os.path.join(OUTPUT_DIR, 'price_vs_intensity.png'), width=900, height=600)            print("  ✓ Price vs intensity PNG saved")        except Exception as e:            print(f"  ✗ Could not save PNG: {e}")                # Controlled regression with GDP per capita        if 'gdp_per_capita' in price_analysis_clean.columns:            price_gdp_clean = price_analysis_clean.dropna(subset=['gdp_per_capita'])            if len(price_gdp_clean) > 10:                X_controlled = price_gdp_clean[['electricity_price_usd_kwh', 'gdp_per_capita']].values                y_controlled = price_gdp_clean['energy_intensity'].values                                lr_controlled = LinearRegression()                lr_controlled.fit(X_controlled, y_controlled)                r2_controlled = lr_controlled.score(X_controlled, y_controlled)                                print(f"\n--- Controlled Regression: Price + GDP per Capita ---")                print(f"R²: {r2_controlled:.3f}")                print(f"Price coefficient: {lr_controlled.coef_[0]:.2f}")                print(f"GDP per capita coefficient: {lr_controlled.coef_[1]:.6f}")                print(f"Intercept: {lr_controlled.intercept_:.2f}")                # Part 2 Conclusion        print("\n" + "=" * 80)        print("PART 2 CONCLUSION: Price vs Efficiency")        print("=" * 80)        print(f"""The analysis reveals a {'positive' if corr > 0 else 'negative'} correlation (r={corr:.3f}, p={p_value:.4f}) betweenelectricity prices and energy intensity. This {'supports' if corr < 0 else 'challenges'} the hypothesis that higherprices drive efficiency improvements. The relationship explains {r2_simple:.1%} of the variance in energyintensity, suggesting that price signals alone are {'not sufficient' if r2_simple < 0.3 else 'important'} fordriving energy efficiency. Other factors such as GDP, industrial structure, climate, and policyframeworks likely play significant roles in determining national energy intensity.""")    else:        print("Insufficient data for price vs efficiency analysis")except Exception as e:    print(f"Error in Part 2 analysis: {e}")    import traceback    traceback.print_exc()

## Part 3: Leapfrogging Analysis

In [ ]:
print("\n" + "=" * 80)print("PART 3: THE DEVELOPMENT HYPOTHESIS - LEAPFROGGING")print("=" * 80)print()try:    # Identify developing economies cluster (Emerging Transitioners or Energy Deficient)    developing_clusters = [k for k, v in cluster_labels_map.items()                           if 'Emerging' in v or 'Deficient' in v]        if developing_clusters:        developing_countries = clustering_clean[clustering_clean['cluster'].isin(developing_clusters)].copy()        print(f"Developing economies identified: {len(developing_countries)} countries")        print(f"Clusters included: {[cluster_labels_map[c] for c in developing_clusters]}")                # Merge with global data for ODA/aid information        if 'Financial flows to developing countries (US $)' in global_data_latest.columns:            aid_merge = global_data_latest[['Entity', 'Financial flows to developing countries (US $)']].copy()            aid_merge.columns = ['country', 'financial_aid_usd']            developing_countries = developing_countries.merge(aid_merge, on='country', how='left')                        # Also get GDP growth            if 'gdp_growth' in global_data_latest.columns:                growth_merge = global_data_latest[['Entity', 'gdp_growth']].copy()                growth_merge.columns = ['country', 'gdp_growth']                developing_countries = developing_countries.merge(growth_merge, on='country', how='left')        else:            print("Financial aid data not available in global dataset")            developing_countries['financial_aid_usd'] = np.nan            developing_countries['gdp_growth'] = np.nan                # Analysis 1: Aid vs Renewable Share        aid_renewable = developing_countries.dropna(subset=['financial_aid_usd', 'renewable_share'])                if len(aid_renewable) > 5:            corr_aid, p_aid = pearsonr(aid_renewable['financial_aid_usd'], aid_renewable['renewable_share'])                        print(f"\n--- Aid vs Renewable Share ---")            print(f"Countries with aid data: {len(aid_renewable)}")            print(f"Correlation: {corr_aid:.3f} (p={p_aid:.4f})")                        # Visualization            fig = px.scatter(                aid_renewable,                x='financial_aid_usd',                y='renewable_share',                hover_data=['country'],                title=f'Financial Aid vs Renewable Energy Share (Developing Economies)\nCorrelation: {corr_aid:.3f}, p={p_aid:.4f}',                labels={'financial_aid_usd': 'Financial Aid (USD)',                        'renewable_share': 'Renewable Share (%)'},                trendline='ols',                width=900,                height=600            )            fig.write_html(os.path.join(OUTPUT_DIR, 'aid_vs_renewable.html'))            print("  ✓ Aid vs renewable plot saved")                        try:                fig.write_image(os.path.join(OUTPUT_DIR, 'aid_vs_renewable.png'), width=900, height=600)                print("  ✓ Aid vs renewable PNG saved")            except Exception as e:                print(f"  ✗ Could not save PNG: {e}")        else:            print("Insufficient data for aid vs renewable analysis")            corr_aid, p_aid = np.nan, np.nan                # Analysis 2: GDP Growth vs Renewable Adoption        growth_renewable = developing_countries.dropna(subset=['gdp_growth', 'renewable_share'])                if len(growth_renewable) > 5:            corr_growth, p_growth = pearsonr(growth_renewable['gdp_growth'], growth_renewable['renewable_share'])                        print(f"\n--- GDP Growth vs Renewable Share ---")            print(f"Countries with growth data: {len(growth_renewable)}")            print(f"Correlation: {corr_growth:.3f} (p={p_growth:.4f})")                        # Visualization            fig = px.scatter(                growth_renewable,                x='gdp_growth',                y='renewable_share',                hover_data=['country'],                title=f'GDP Growth vs Renewable Energy Share (Developing Economies)\nCorrelation: {corr_growth:.3f}, p={p_growth:.4f}',                labels={'gdp_growth': 'GDP Growth (%)',                        'renewable_share': 'Renewable Share (%)'},                trendline='ols',                width=900,                height=600            )            fig.write_html(os.path.join(OUTPUT_DIR, 'gdp_growth_vs_renewable.html'))            print("  ✓ GDP growth vs renewable plot saved")                        try:                fig.write_image(os.path.join(OUTPUT_DIR, 'gdp_growth_vs_renewable.png'), width=900, height=600)                print("  ✓ GDP growth vs renewable PNG saved")            except Exception as e:                print(f"  ✗ Could not save PNG: {e}")        else:            print("Insufficient data for GDP growth vs renewable analysis")            corr_growth, p_growth = np.nan, np.nan                # Part 3 Conclusion        print("\n" + "=" * 80)        print("PART 3 CONCLUSION: Leapfrogging in Developing Economies")        print("=" * 80)        if not np.isnan(corr_aid):            print(f"""The leapfrogging analysis focused on {len(developing_countries)} developing economies.Financial Aid & Renewables: {f'Correlation of {corr_aid:.3f} (p={p_aid:.4f})' if not np.isnan(corr_aid) else 'Insufficient data'}{'A positive correlation suggests that financial aid may support renewable energy adoption.' if corr_aid > 0.2 and p_aid < 0.05 else 'No strong evidence that financial aid directly drives renewable adoption.'}GDP Growth & Renewables: {f'Correlation of {corr_growth:.3f} (p={p_growth:.4f})' if not np.isnan(corr_growth) else 'Insufficient data'}{'Growing economies show higher renewable adoption, supporting the leapfrogging hypothesis.' if corr_growth > 0.2 and p_growth < 0.05 else 'No clear link between economic growth and renewable adoption.'}The leapfrogging hypothesis {'receives support' if (corr_aid > 0.2 or corr_growth > 0.2) else 'is not strongly supported'}from this data. Developing countries may benefit from targeted financial assistance andfavorable economic conditions to accelerate renewable energy deployment.""")        else:            print("\nInsufficient data to draw strong conclusions about leapfrogging in developing economies.")    else:        print("No developing economies cluster identified")        except Exception as e:    print(f"Error in Part 3 analysis: {e}")    import traceback    traceback.print_exc()

## Part 4: Momentum Analysis

In [ ]:
print("\n" + "=" * 80)print("PART 4: THE TEMPORAL DYNAMIC - MOMENTUM ANALYSIS")print("=" * 80)print()try:    # Filter OWID data for 2010-2020    start_year = 2010    end_year = 2020        owid_temporal = owid_data[(owid_data['year'] >= start_year) &                                (owid_data['year'] <= end_year)].copy()        print(f"Temporal data: {start_year}-{end_year}")    print(f"Total observations: {len(owid_temporal)}")        # Calculate renewable share and CO2 intensity    if 'renewables_share_energy' in owid_temporal.columns:        owid_temporal['renewable_share'] = owid_temporal['renewables_share_energy']    elif 'renewables_electricity' in owid_temporal.columns and 'electricity_generation' in owid_temporal.columns:        owid_temporal['renewable_share'] = (            owid_temporal['renewables_electricity'] / owid_temporal['electricity_generation'] * 100        )    else:        owid_temporal['renewable_share'] = np.nan        if 'carbon_intensity_elec' in owid_temporal.columns:        owid_temporal['co2_intensity'] = owid_temporal['carbon_intensity_elec']    elif 'co2' in owid_temporal.columns and 'primary_energy_consumption' in owid_temporal.columns:        owid_temporal['co2_intensity'] = (            owid_temporal['co2'] / owid_temporal['primary_energy_consumption']        )    else:        owid_temporal['co2_intensity'] = np.nan        # Compute per-country slopes using linear regression    momentum_results = []        countries = owid_temporal['country'].unique()    print(f"Analyzing momentum for {len(countries)} countries...")        for country in countries:        country_data = owid_temporal[owid_temporal['country'] == country].sort_values('year')                # Need at least 5 years of data        if len(country_data) < 5:            continue                iso_code = country_data['iso_code'].iloc[0] if 'iso_code' in country_data.columns else None                # Renewable share momentum        renewable_data = country_data.dropna(subset=['renewable_share'])        if len(renewable_data) >= 5:            X = renewable_data['year'].values.reshape(-1, 1)            y = renewable_data['renewable_share'].values            lr = LinearRegression()            lr.fit(X, y)            renewable_slope = lr.coef_[0]            renewable_delta = renewable_data['renewable_share'].iloc[-1] - renewable_data['renewable_share'].iloc[0]        else:            renewable_slope = np.nan            renewable_delta = np.nan                # CO2 intensity momentum        co2_data = country_data.dropna(subset=['co2_intensity'])        if len(co2_data) >= 5:            X = co2_data['year'].values.reshape(-1, 1)            y = co2_data['co2_intensity'].values            lr = LinearRegression()            lr.fit(X, y)            co2_slope = lr.coef_[0]            co2_delta = co2_data['co2_intensity'].iloc[-1] - co2_data['co2_intensity'].iloc[0]        else:            co2_slope = np.nan            co2_delta = np.nan                momentum_results.append({            'country': country,            'iso_code': iso_code,            'renewable_slope': renewable_slope,            'renewable_delta': renewable_delta,            'co2_slope': co2_slope,            'co2_delta': co2_delta        })        momentum_df = pd.DataFrame(momentum_results)    print(f"Countries with momentum data: {len(momentum_df)}")        # Classify countries based on momentum    def classify_momentum(row):        """Classify countries into momentum categories"""        renewable_slope = row['renewable_slope']        co2_slope = row['co2_slope']                if np.isnan(renewable_slope):            return 'Insufficient Data'                if renewable_slope >= MOMENTUM_THRESHOLDS['renewable_slope_high']:            if not np.isnan(co2_slope) and co2_slope < MOMENTUM_THRESHOLDS['co2_slope_high']:                return 'Accelerating Adopters'            else:                return 'Accelerating Adopters'        elif renewable_slope >= MOMENTUM_THRESHOLDS['renewable_slope_low']:            return 'Legacy Green'        else:            return 'Falling Behind'        momentum_df['momentum_category'] = momentum_df.apply(classify_momentum, axis=1)        print("\n--- Momentum Classification ---")    print(momentum_df['momentum_category'].value_counts())        # Save momentum analysis    output_path = os.path.join(OUTPUT_DIR, 'momentum_analysis_2010_2020.csv')    momentum_df.to_csv(output_path, index=False)    print(f"\n✓ Saved momentum analysis to {output_path}")        # Visualization 1: Histogram of renewable slopes    fig, axes = plt.subplots(1, 2, figsize=(14, 5))        momentum_clean = momentum_df.dropna(subset=['renewable_slope'])    axes[0].hist(momentum_clean['renewable_slope'], bins=30, edgecolor='black', alpha=0.7)    axes[0].axvline(MOMENTUM_THRESHOLDS['renewable_slope_high'], color='red', linestyle='--',                     label=f'High threshold ({MOMENTUM_THRESHOLDS["renewable_slope_high"]})')    axes[0].axvline(MOMENTUM_THRESHOLDS['renewable_slope_low'], color='orange', linestyle='--',                    label=f'Low threshold ({MOMENTUM_THRESHOLDS["renewable_slope_low"]})')    axes[0].set_xlabel('Renewable Share Slope (%/year)')    axes[0].set_ylabel('Number of Countries')    axes[0].set_title('Distribution of Renewable Energy Momentum (2010-2020)')    axes[0].legend()    axes[0].grid(alpha=0.3)        # Histogram of CO2 slopes    momentum_co2_clean = momentum_df.dropna(subset=['co2_slope'])    axes[1].hist(momentum_co2_clean['co2_slope'], bins=30, edgecolor='black', alpha=0.7, color='coral')    axes[1].axvline(MOMENTUM_THRESHOLDS['co2_slope_high'], color='green', linestyle='--',                    label=f'Reduction threshold ({MOMENTUM_THRESHOLDS["co2_slope_high"]})')    axes[1].set_xlabel('CO2 Intensity Slope (change/year)')    axes[1].set_ylabel('Number of Countries')    axes[1].set_title('Distribution of CO2 Intensity Momentum (2010-2020)')    axes[1].legend()    axes[1].grid(alpha=0.3)        plt.tight_layout()    plt.savefig(os.path.join(OUTPUT_DIR, 'momentum_histograms.png'), dpi=150, bbox_inches='tight')    print("  ✓ Momentum histograms saved")    plt.close()        # Visualization 2: Scatter plot of renewable slope vs CO2 slope    momentum_scatter = momentum_df.dropna(subset=['renewable_slope', 'co2_slope'])        fig = px.scatter(        momentum_scatter,        x='renewable_slope',        y='co2_slope',        color='momentum_category',        hover_data=['country'],        title='Renewable Energy Momentum vs CO2 Intensity Change (2010-2020)',        labels={'renewable_slope': 'Renewable Share Slope (%/year)',                'co2_slope': 'CO2 Intensity Slope (change/year)'},        width=1000,        height=600    )    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="No CO2 change")    fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="No renewable change")    fig.write_html(os.path.join(OUTPUT_DIR, 'momentum_scatter.html'))    print("  ✓ Momentum scatter plot saved")        try:        fig.write_image(os.path.join(OUTPUT_DIR, 'momentum_scatter.png'), width=1000, height=600)        print("  ✓ Momentum scatter PNG saved")    except Exception as e:        print(f"  ✗ Could not save PNG: {e}")        # Part 4 Conclusion    print("\n" + "=" * 80)    print("PART 4 CONCLUSION: Momentum Analysis")    print("=" * 80)        category_counts = momentum_df['momentum_category'].value_counts()    total = category_counts.sum()        print(f"""The temporal analysis (2010-2020) reveals diverse momentum patterns across {len(momentum_df)} countries:""")    for category, count in category_counts.items():        pct = count / total * 100        print(f"{category}: {count} countries ({pct:.1f}%)")        print(f"""Accelerating Adopters show renewable energy growth exceeding {MOMENTUM_THRESHOLDS['renewable_slope_high']}%/year,indicating rapid energy transitions. Legacy Green countries maintain steady renewable portfolios.Countries Falling Behind show minimal or negative renewable growth, suggesting continued fossilfuel dependence.The distribution of renewable slopes shows that {'most' if (momentum_clean['renewable_slope'] > 0).mean() > 0.6 else 'many'} countries are increasing renewable adoption, though at varying rates. The relationship betweenrenewable momentum and CO2 intensity changes {'shows' if momentum_scatter[['renewable_slope', 'co2_slope']].corr().iloc[0, 1] < -0.3 else 'does not clearly show'} that accelerating renewable adoption drives emissions reductions.This momentum analysis provides actionable insights for policy: countries should aim for renewablegrowth rates above {MOMENTUM_THRESHOLDS['renewable_slope_high']}%/year to achieve meaningful decarbonization by 2050.""")    except Exception as e:    print(f"Error in Part 4 analysis: {e}")    import traceback    traceback.print_exc()